In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, precision_score
from sklearn.feature_selection import SelectFromModel

# Preprocessing Pipeline Plan

- Limit observations to only the first encounter
- Drop additional prescribed medication due to limited clinical relevance
- Conduct feature decomposition on insulin and metformin to include a binary flag for being prescribed the medication and an ordinal encoding for the change
- Create train, validation and test splits
- Preprocess features
    - Convert data types (admission source, admission type)
    - Scale numeric features
    - Encode categoricals (OHE and ordinal)
    - Encode target (multi-class, binary)

## Experimenting with Feature Decomposition

Decomposing the insulin and metformin columns into two each: one binary flag to indicate if someone is on the drug at all, another ordinal encoding encoding if the patient is on the drug to indicate if there was a change in dosing.

In [2]:
features = pd.read_parquet("data/diabetes_features.parquet")
target = pd.read_parquet("data/diabetes_target.parquet")

### Limiting the Data to the First Encounter Only

In [3]:
# Creating a dataframe for selecting first encounters
splitting_df = features.copy()
splitting_df['target'] = (target['readmitted'] == "<30").astype(int)

# Keeping only the first encounter per patient_nbr
split_df = splitting_df.drop_duplicates('patient_nbr', keep='first')

### Preparing Features

Binary and Ordinal Splits for Insulin and Metformin

In [4]:
# Creating a feature to indicate if a patient is not on insulin or metformin
split_df['missing_insulin'] = (split_df.insulin == "No").astype(int)
split_df['missing_metformin'] = (split_df.metformin == "No").astype(int)

# Creating a dictionary to encode changes in insulin and metformin
encode_dict = {'No': 0,
               "Steady": 0,
               "Up": 1,
               "Down": -1}

# Applying the encoding to engineer new features
split_df['metformin_change'] = split_df['metformin'].apply(lambda x: encode_dict[x])
split_df['insulin_change'] = split_df['insulin'].apply(lambda x: encode_dict[x])

Binary missingness encoding for weight

In [5]:
split_df['missing_weight'] = (split_df['weight'] == "?").astype("int")

Binary indicator for missingness to one-hot encode:
- medical_specialty
- payer_code

In [6]:
# Utility function for encoding missing values in a column for one-hot encoding
def missing_cleaner(x, missing_code, encoding=""):
    if x == missing_code:
        return encoding
    else:
        return x

split_df['payer_code_cleaned'] = split_df['payer_code'].apply(missing_cleaner, args=("?", "missing_payer"))
split_df['medical_specialty_cleaned'] = split_df['medical_specialty'].apply(missing_cleaner, args=("?", "missing_medical_specialty"))

Binary indicator and ordinal encoding for A1C and max_glu_serum

In [7]:
split_df['missing_a1c'] = (split_df['A1Cresult'] == "None").astype("int")
split_df['missing_max_glu_serum'] = (split_df['max_glu_serum'] == "None").astype("int")

#### Creating Train and Test Splits

In [8]:
# List of features to drop
drop_features = ['patient_nbr', 'encounter_id', 'target',
                 'repaglinide', 'nateglinide', 'chlorpropamide',
                 'glimepiride', 'acetohexamide', 'glipizide',
                 'glyburide', 'tolbutamide',
                 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
                 'tolazamide', 'examide', 'citoglipton', 'insulin',
                 'glyburide.metformin', 'glipizide.metformin',
                 'glimepiride.pioglitazone', 'metformin.rosiglitazone',
                 'metformin.pioglitazone', 'metformin', 'insulin',
                 'A1Cresult', 'weight', 'diag_1', 'diag_2', 'diag_3',
                 'max_glu_serum', 'change']

# Building X & y dataframes
y = split_df['target']
X = split_df.drop(columns=drop_features)

# Creating train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

## Setting up the Pipeline

Features for scaling:
- time_in_hospital
- num_lab_procedures
- num_procedures
- num_medications
- number_outpatient
- number_inpatient
- number_diagnoses

Features for OHE:
- race
- gender
- admission_type_id
- dicharge_disposition_id
- admission_source_id
- payer_code
- medical_specialty

Features for ordinal encoding:
- Age
- Metformin (already encoded)
- Insulin (already encoded)

#### OHE Columns

In [18]:
OHEFEATURES = ['race', 'gender', 'admission_type_id',
               'discharge_disposition_id', 'admission_source_id',
               'payer_code', 'medical_specialty']

# Pipeline for OHE
ohe_pipeline = Pipeline([('handle_outliers', SimpleImputer(strategy='most_frequent')),
                         ('encoding', OneHotEncoder(handle_unknown='ignore',
                                                    min_frequency=0.01,
                                                    ).set_output(transform='default'))])

#### Columns for Scaling

In [19]:
SCALING_FEATURES = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                    'num_medications', 'number_outpatient', 'number_inpatient', 'number_diagnoses']

# Pipeline for Scaling
scaling_pipeline = Pipeline([('scaling', MinMaxScaler())])

#### Columns for Ordinal Encoding

In [20]:
# Setting up a list of columns for ordinal encoding
ORDINAL_COLS = ['age']

# Creating a list of categories for the age column
age_categories = sorted(split_df['age'].unique())

# Setting up a pipeline for ordinal encoding
ordinal_pipeline = Pipeline([('ordinal_encoding', OrdinalEncoder(categories=[age_categories]))])

#### Setting up the Column Transformer

In [ ]:
ct = ColumnTransformer([('OHE', ohe_pipeline, OHEFEATURES),
                        ('scaling', scaling_pipeline, SCALING_FEATURES),
                        ('ordinal_encoding', ordinal_pipeline, ORDINAL_COLS)])

# Setting the output to return a pandas dataset
ct.set_output(default='pandas')

## Fitting the Model

For next session: dive deeper into understanding analytics with imbalanced data

In [44]:
# Setting up a dictionary of models
models = {'random_forest': RandomForestClassifier(class_weight='balanced'),
          'log_reg': LogisticRegression()}

# Creating the parameter grid
param_grid = {
    'random_forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 5, 10],
        'model__min_samples_split': [2, 5]
    },
    'log_reg': {
        'model__C': np.logspace(-3, 2, 6)
    }
}

# Setting up a scoring dictionary
scoring_dict = {
    'average_precision': 'average_precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc',
    'precision': make_scorer(precision_score, zero_division=0)
    }

In [45]:
cv = StratifiedKFold(n_splits=5,
                     random_state=42,
                     shuffle=True)

In [46]:
cv_results = {}

# Setting up the loop to evaluate the different models across the parameter grids
for model_name, model in models.items():

    # Pipeline for model training
    model_training_pipeline = Pipeline([('preprocessing', ct),
                                        ('model', model)])

    # Setting up the grid for CV
    grid = GridSearchCV(model_training_pipeline,
                        param_grid=param_grid[model_name],
                        scoring=scoring_dict,
                        refit='average_precision',
                        cv=cv)

    # Storing the results
    cv_results[model_name] = grid.fit(X_train, y_train)

/Users/samueljoseph/Documents/Programming/Diabetes/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/samueljoseph/Documents/Programming/Diabetes/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as s

In [47]:
results_df = pd.concat(
    {name: pd.DataFrame(grid.cv_results_) for name, grid in cv_results.items()},
    names=['model']
).reset_index(level='model')

# The columns you actually care about:
cols = ['model', 'params', 'mean_test_average_precision',
        'mean_test_recall', 'mean_test_roc_auc', 'rank_test_average_precision']
results_df[cols].sort_values('mean_test_average_precision', ascending=False)

,model,params,mean_test_average_precision,mean_test_recall,mean_test_roc_auc,rank_test_average_precision
11,random_forest,"{'model__max_depth': 10, 'model__min_samples_s...",0.170077,0.476898,0.647274,1
10,random_forest,"{'model__max_depth': 10, 'model__min_samples_s...",0.170041,0.479047,0.645660,2
9,random_forest,"{'model__max_depth': 10, 'model__min_samples_s...",0.169341,0.480798,0.647153,3
8,random_forest,"{'model__max_depth': 10, 'model__min_samples_s...",0.168923,0.470661,0.645310,4
5,log_reg,{'model__C': 100.0},0.166353,0.003704,0.644618,1
4,log_reg,{'model__C': 10.0},0.166273,0.003704,0.644738,2
3,log_reg,{'model__C': 1.0},0.166105,0.002730,0.644471,3
5,random_forest,"{'model__max_depth': 5, 'model__min_samples_sp...",0.165187,0.566002,0.645127,5
7,random_forest,"{'model__max_depth': 5, 'model__min_samples_sp...",0.165119,0.570486,0.645314,6
6,random_forest,"{'model__max_depth': 5, 'model__min_samples_sp...",0.164776,0.570679,0.644410,7
